## Concept 7 — Plan-and-Execute Agent

### What is it?
For complex multi-step queries, ReAct can lose track of the goal.
Plan-and-Execute separates work into three roles:
1. **Planner** — generates a step-by-step plan upfront
2. **Executor** — executes each step using tools, passing results forward
3. **Synthesizer** — combines all step results into a final answer

### Pipeline
```
Query
  → Planner LLM  → [Step 1, Step 2, Step 3]
  → Executor     → Step 1 result
  → Executor     → Step 2 result (may use Step 1 result)
  → Executor     → Step 3 result
  → Synthesizer  → Final coherent answer
```

### Limitation (what Concept 8 fixes)
No quality validation — the final answer isn't checked for correctness.
→ Concept 8 adds a critic LLM that validates and triggers a retry if needed.

In [ ]:
print('Hi')

In [ ]:
import sys
sys.path.insert(0, '.')

from AgentUtils import get_llm, TOOLS, TEST_QUERIES, run_queries
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm      = get_llm(temperature=0.0)
tool_map = {t.name: t for t in TOOLS}
print(f'LLM ready | Tools: {list(tool_map.keys())}')

### Step 1 — Planner
The Planner LLM decomposes the query into an ordered list of concrete steps.

In [ ]:
planner_prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are a planning assistant. Break the user query into clear, ordered steps.\n'
     'Available tools: calculate (math), search_docs (LangChain docs), get_weather (weather).\n'
     'Output ONLY a numbered list. Each step should be one concrete action.\n'
     'Example:\n'
     '1. Call search_docs to find information about RAG\n'
     '2. Call calculate to compute 25 * 4\n'
     '3. Combine results into a final answer'),
    ('human', 'Query: {query}\n\nCreate the plan:'),
])
planner_chain = planner_prompt | llm | StrOutputParser()

# Test planner
test_query = 'Search docs for RAG and also calculate 25 multiplied by 4'
plan = planner_chain.invoke({'query': test_query})
print(f'Query: {test_query}\n')
print(f'Plan:\n{plan}')

### Step 2 — Executor
The Executor runs one step at a time, passing previous results as context.

In [ ]:
llm_with_tools = llm.bind_tools(TOOLS)

def execute_step(step: str, context: str) -> str:
    step_prompt = (
        f'Execute this step using the available tools.\n'
        f'Step: {step}\n'
        f'Context from previous steps: {context if context else "None"}\n'
        f'Complete this step and return the result.'
    )
    messages = [HumanMessage(content=step_prompt)]
    response = llm_with_tools.invoke(messages)
    messages.append(response)

    if response.tool_calls:
        for call in response.tool_calls:
            result = tool_map[call['name']].invoke(call['args'])
            print(f'    [Tool] {call["name"]}({call["args"]}) → {result}')
            messages.append(ToolMessage(content=str(result), tool_call_id=call['id']))
        final = llm_with_tools.invoke(messages)
        return final.content

    return response.content

# Test executor on step 1 of our plan
steps = [line.strip() for line in plan.strip().split('\n') if line.strip() and line[0].isdigit()]
print(f'Parsed {len(steps)} steps:')
for s in steps:
    print(f'  {s}')

print(f'\nExecuting step 1:')
r = execute_step(steps[0], '')
print(f'Result: {r}')

### Step 3 — Synthesizer
After all steps complete, the Synthesizer combines results into one coherent answer.

In [ ]:
synthesizer_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You synthesize step results into a clear, complete final answer.'),
    ('human',
     'Original query: {query}\n\n'
     'Step results:\n{step_results}\n\n'
     'Write the final answer:'),
])
synthesizer_chain = synthesizer_prompt | llm | StrOutputParser()

### Step 4 — Full Plan-and-Execute Loop
See all three roles working together on a complex query.

In [ ]:
def plan_and_execute(query: str) -> str:
    print(f'\n[Planner] Generating plan...')
    plan_text = planner_chain.invoke({'query': query})

    # Parse numbered steps
    steps = [l.strip() for l in plan_text.strip().split('\n')
             if l.strip() and (l.strip()[0].isdigit() or l.strip().startswith('-'))]
    if not steps:
        steps = [plan_text]  # fallback if parsing fails

    print(f'[Planner] {len(steps)} steps planned')

    # Execute each step
    step_results = []
    context      = ''

    for i, step in enumerate(steps, 1):
        print(f'\n[Executor] Step {i}: {step}')
        result = execute_step(step, context)
        step_results.append(f'Step {i} ({step}): {result}')
        context = '\n'.join(step_results)
        print(f'  Result: {result[:100]}...' if len(result) > 100 else f'  Result: {result}')

    # Synthesize
    print(f'\n[Synthesizer] Combining {len(step_results)} results...')
    final = synthesizer_chain.invoke({
        'query':        query,
        'step_results': context,
    })
    return final

# Run a complex multi-step query
complex_q = 'Search docs for RAG and also calculate 25 multiplied by 4'
print('=== Plan-and-Execute Demo ===')
answer = plan_and_execute(complex_q)
print(f'\n=== FINAL ANSWER ===\n{answer}')

### Full Function

In [ ]:
def plan_execute_agent(query: str) -> str:
    return plan_and_execute(query)

### Test — Standard Queries

In [ ]:
run_queries(plan_execute_agent, label='Concept 7 — Plan-and-Execute Agent')